# Volume 14, Chapter 4: The anatomy of integers (Python companion)

Exercises first, solutions at the end. We count smooth numbers and test Rankin's bound, factor integers with Dixon's random squares, watch the arcsine law of a random divisor appear (slowly), simulate the broken stick of Lemma 4.7, and count the distinct entries of the multiplication table. The computer science application is **Dixon's factoring algorithm**, with Gaussian elimination over $\mathbb F_2$ on bitsets. Requires `numpy` and `scipy`.

In [ ]:
import math, time, random
import numpy as np
from scipy.optimize import minimize_scalar

def primes_upto(N):
    s = np.ones(N + 1, dtype=bool); s[:2] = False
    for i in range(2, int(N ** 0.5) + 1):
        if s[i]: s[i * i::i] = False
    return np.nonzero(s)[0]

## Exercises

**P4.1 (★) Smooth numbers up to $10^7$.** Compute the largest prime factor of every $n\le10^7$ by a sieve, then $\Psi(10^7,y)$ for $y=10,100,1000,10^4$. Compare with Rankin's bound minimised over $\sigma$ (Proposition 4.1), with the lower bound $\binom{\pi(y)}{\lfloor u\rfloor}$ (Proposition 4.3) and with $10^7\rho(u)$ (Figure 4.1).

**P4.2 (★★) Rankin's choice of $\sigma$.** For $x=10^{30}$ and $y=10^3,10^4,10^5,10^6$, find the $\sigma$ minimising Rankin's bound numerically and compare it with $1-\frac{\log(u\log u)}{\log y}$, the choice in the proof of Theorem 4.2.

**P4.3 (★★) Dixon's algorithm.** Implement the algorithm of Section 4.2 (trial division of random squares, then Gaussian elimination over $\mathbb F_2$ with Python integers as bit vectors) and factor $n=100003\cdot1000003$ and a product of two random $7$-digit primes. Count the draws, compare the observed proportion of smooth squares with $\rho(u)$, $u=\frac{\log n}{\log y}$, and with the proven lower bound of the proof of Theorem 4.6.

**P4.4 (★) A random divisor.** For $n\le10^6$, compute the law of $\frac{\log d}{\log n}$ with $d$ a uniform divisor of $n$, its Kolmogorov distance to the arcsine law, and $P(d=1)=\mathbb E\frac1{\tau(n)}$, compared with $\frac{C}{\sqrt{\log x}}$ of Exercise 4.5.

**P4.5 (★★) The broken stick.** Simulate $S=\sum_i\varepsilon_iV_i$ (GEM sticks, truncated when the remaining length is below $10^{-12}$) with $\varepsilon_i$ Bernoulli$(\theta)$, for $\theta=\frac12$ and $\theta=0.3$, and compare the moments with $\binom{2m}m4^{-m}$ (Lemma 4.7) and $\frac{\theta(\theta+1)\cdots(\theta+m-1)}{m!}$ (Exercise 4.6).

**P4.6 (★) The multiplication table.** Compute $D(N)$ for $N=10$ and $N=2^k$, $k\le12$, and the local exponent $-\frac{\log(D(2N)/4N^2)-\log(D(N)/N^2)}{\log\log2N-\log\log N}$. Is it close to $\delta=0.086$? What does Ford's order of magnitude $(\log N)^{-\delta}(\log\log N)^{-3/2}$ predict for it?

## Solutions

### P4.1

In [ ]:
N = 10**7
P = primes_upto(N)
lpf = np.ones(N + 1, dtype=np.int32)
for p in P: lpf[p::p] = p          # the last prime written is the largest
Psi = np.cumsum(np.bincount(lpf[1:], minlength=N + 1))

du = 1e-3; us = np.arange(0, 12, du); rho = np.ones(len(us))
m = int(round(1 / du)); cs = np.zeros(len(us) + 1); cs[1:m + 1] = np.cumsum(rho[:m]) * du
for i in range(m, len(us)):        # u rho(u) = integral of rho over [u-1, u]
    rho[i] = (cs[i] - cs[i - m]) / us[i]; cs[i + 1] = cs[i] + rho[i] * du

lx = math.log(N)
for y in [10, 100, 1000, 10**4]:
    Pp = P[P <= y].astype(float); u = lx / math.log(y)
    f = lambda s: s * lx - np.sum(np.log1p(-Pp ** (-s)))
    r = minimize_scalar(f, bounds=(0.05, 1.0), method='bounded')
    low = math.comb(len(Pp), int(u))
    print(f"y={y:6d} u={u:5.2f}  Psi={Psi[y]:8d}  Rankin={math.exp(r.fun):12.0f} (sigma={r.x:.3f})"
          f"  lower={low:8d}  x*rho(u)={N * np.interp(u, us, rho):10.1f}")

Rankin's bound is $3$ to $25$ times too large, but has the right shape at every $y$. Dickman's $\rho$ is excellent for $y\ge100$ and useless for $y=10$, where $u=7$: the $2155$ integers that are $7$-smooth are counted by lattice points (Exercise 4.1), not by $\rho$. The lower bound is crude when $\lfloor u\rfloor$ is $1$ or $2$.

### P4.2

In [ ]:
lx = 30 * math.log(10)
for y in [10**3, 10**4, 10**5, 10**6]:
    Pp = primes_upto(y).astype(float); u = lx / math.log(y)
    f = lambda s: s * lx - np.sum(np.log1p(-Pp ** (-s)))
    r = minimize_scalar(f, bounds=(0.05, 1.0), method='bounded')
    s_proof = 1 - math.log(u * math.log(u)) / math.log(y)
    print(f"y=10^{round(math.log10(y))}  u={u:5.2f}  best sigma={r.x:.4f}  proof sigma={s_proof:.4f}"
          f"  log10 bound: best {r.fun / math.log(10):.2f}, proof {f(s_proof) / math.log(10):.2f}")

The proof's $\sigma$ is close to the optimum and loses little: the saddle point of $\sigma\mapsto\sigma\log x+\sum_p-\log(1-p^{-\sigma})$ solves $\sum_{p\le y}\frac{\log p}{p^\sigma-1}=\log x$, and for $\sigma$ near $1$ the left side is about $\frac{y^{1-\sigma}}{1-\sigma}$, which gives $y^{1-\sigma}\approx u\log u$.

### P4.3

In [ ]:
def dixon(n, y, rng):
    qs = [int(q) for q in primes_upto(y)]
    for q in qs:
        if n % q == 0: return q, 0, 0
    h = len(qs); rels = []; draws = 0
    while len(rels) < h + 1:
        x = rng.randrange(1, n); draws += 1
        g = math.gcd(x, n)
        if g > 1: return g, draws, len(rels)
        s = x * x % n; t = s; e = [0] * h
        for j, q in enumerate(qs):
            while t % q == 0: t //= q; e[j] += 1
        if t == 1: rels.append((x, e))
    # Gaussian elimination over F_2: rows are bitsets, 'comb' records which relations were added
    rows = [(sum((ej & 1) << j for j, ej in enumerate(e)), 1 << i) for i, (x, e) in enumerate(rels)]
    pivots = {}
    for v, comb in rows:
        while v:
            b = v.bit_length() - 1
            if b not in pivots: pivots[b] = (v, comb); break
            pv, pc = pivots[b]; v ^= pv; comb ^= pc
        if v == 0:                       # a dependency: product of these s_i is a square
            I = [i for i in range(len(rels)) if comb >> i & 1]
            X = 1; tot = [0] * h
            for i in I:
                X = X * rels[i][0] % n
                tot = [a + b for a, b in zip(tot, rels[i][1])]
            Z = 1
            for q, t2 in zip(qs, tot): Z = Z * pow(q, t2 // 2, n) % n
            g = math.gcd(X - Z, n)
            if 1 < g < n: return g, draws, len(rels)
    return None, draws, len(rels)

rng = random.Random(2026)
def rand_prime(lo, hi):
    while True:
        p = rng.randrange(lo, hi) | 1
        if all(p % d for d in range(3, int(p ** 0.5) + 1, 2)): return p
for n in [100003 * 1000003, rand_prime(10**6, 10**7) * rand_prime(10**6, 10**7)]:
    y = 200
    t = time.time(); g, draws, nrel = dixon(n, y, rng); dt = time.time() - t
    u = math.log(n) / math.log(y)
    print(f"n={n}: factor {g} (cofactor {n // g if g else None}), {draws} draws for {nrel} relations, {dt:.1f}s")
    print(f"   smooth proportion {nrel / draws:.2e},  rho(u={u:.2f}) = {np.interp(u, us, rho):.2e}")

The observed proportion of smooth squares is $2$ to $3$ times $\rho(u)$. The residues $x^2\bmod n$ behave like random integers below $n$, as far as smoothness is concerned, and for integers of this size $\Psi(x,y)/x$ is still noticeably larger than its limit $\rho(u)$ (compare P4.1 at $y=100$, where the ratio is $1.7$). The proven bound $y^{-2}(2\log n)^{-u}$ of Theorem 4.6 is many orders of magnitude smaller; it is a worst case that only uses squares of products of distinct small primes. If the final gcd is trivial (probability at most $\frac12$), the algorithm would simply restart; here it returns `None` in that case.

### P4.4

In [ ]:
N = 10**6
tau = np.zeros(N + 1)
for d in range(1, N + 1): tau[d::d] += 1
w = np.zeros(N + 1); w[1:] = 1 / tau[1:]
logn = np.zeros(N + 1); logn[2:] = np.log(np.arange(2, N + 1))
B = 200; hist = np.zeros(B)
for d in range(1, N + 1):
    n = np.arange(d, N + 1, d)
    if d == 1: n = n[1:]
    t = math.log(d) / logn[n]
    hist += np.bincount(np.minimum((t * B).astype(int), B - 1), weights=w[n], minlength=B)
Fd = np.cumsum(hist) / hist.sum()
ts = np.arange(1, B + 1) / B
print("Kolmogorov distance to arcsine:", np.abs(Fd - 2 / math.pi * np.arcsin(np.sqrt(ts))).max())
Pr = P[P < 10**6].astype(float)
Cc = np.prod(np.sqrt(1 - 1 / Pr) * Pr * np.log(Pr / (Pr - 1))) / math.sqrt(math.pi)
print("P(d=1) =", w[1:].mean(), " C/sqrt(log x) =", Cc / math.sqrt(math.log(N)), " C =", Cc)

The distance is about $0.1$, dominated by the atoms at $d=1$ and $d=n$, of mass $\mathbb E\frac1{\tau(n)}\approx\frac{0.547}{\sqrt{\log x}}$ each. The arcsine law is reached only at the speed $(\log x)^{-1/2}$: true theorems about random divisors are hard to see on a computer.

### P4.5

In [ ]:
rng2 = np.random.default_rng(4)
def sample_S(theta, size):
    S = np.zeros(size); W = np.ones(size)
    while W.max() > 1e-12:
        U = rng2.random(size); V = U * W; W = W * (1 - U)
        S += V * (rng2.random(size) < theta)
    return S
for theta in [0.5, 0.3]:
    S = sample_S(theta, 200000)
    for m in [1, 2, 3, 4]:
        exact = math.prod(theta + i for i in range(m)) / math.factorial(m)
        print(f"theta={theta} m={m}: simulated {np.mean(S**m):.4f}  exact {exact:.4f}")
print("arcsine moments:", [math.comb(2 * m, m) / 4**m for m in range(1, 5)])

For $\theta=\frac12$ the exact moments $\frac{(1/2)(3/2)\cdots}{m!}$ are $\binom{2m}m4^{-m}$: $0.5,0.375,0.3125,0.2734$. The simulation agrees to Monte Carlo accuracy, about $10^{-3}$.

### P4.6

In [ ]:
def D(M):
    arr = np.zeros(M * M + 1, dtype=bool)
    for a in range(1, M + 1): arr[a:a * M + 1:a] = True
    return int(arr.sum())
print("D(10) =", D(10))
delta = 1 - (1 + math.log(math.log(2))) / math.log(2)
prev = None
for k in range(1, 13):
    M = 2**k; r = D(M) / M**2
    if prev:
        loc = -(math.log(r) - math.log(prev[1])) / (math.log(math.log(M)) - math.log(math.log(prev[0])))
        print(f"N=2^{k:2d}  D/N^2={r:.4f}  local exponent {loc:.3f}")
    prev = (M, r)
print("delta =", delta, "  delta + 3/2 / loglog N at N=4096:", delta + 1.5 / math.log(math.log(4096)))

The local exponent wanders between $0.34$ and $0.74$ and settles near $0.35$: far above $\delta=0.086$. Ford's order of magnitude predicts a local exponent $\delta+\frac{3/2}{\log\log N}$, which is $0.79$ at $N=4096$ and decreases to $\delta$ only as $\log\log N\to\infty$; the observed value is smaller still, so at this size neither asymptotic regime has set in (Ford's theorem holds up to an unknown constant factor, which can vary slowly in this range). The dashed curve of Figure 4.3 matches the last points because its constant was fitted there. The exponent $\delta$ is a theorem about astronomically large tables.